In [ ]:
# %%
from pathlib import Path
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.notebook import tqdm

from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------------------------
# Load model
# -------------------------
checkpoint = "./checkpoints/sam2.1_hiera_large.pt"
model_cfg  = "configs/sam2.1/sam2.1_hiera_l.yaml"

sam2_model = build_sam2(model_cfg, checkpoint, device=device)
sam2_model.eval()

predictor = SAM2ImagePredictor(sam2_model)
print("Model loaded on", device)

Model loaded on cuda
2697 2697 2697 2697


In [ ]:

# -------------------------
# Load data
# -------------------------
mask_clean      = np.load("../data/prompts/mask_clean.npy")          # (N,2,H,W) or similar
images_clean    = np.load("../data/prompts/images_clean.npy")        # (N,2,H,W) or similar
centroids_final = np.load("../data/prompts/centroids_final.npy", allow_pickle=True)
valid_idx       = np.load("../data/prompts/valid_idx.npy")

print(len(mask_clean), len(images_clean), len(centroids_final), len(valid_idx))

In [ ]:
# -------------------------
# Point sampling
# -------------------------
def generate_points_around_centroid(
    mask: np.ndarray,
    centroid_xy,
    num_pos_points=8,
    num_neg_points=8,
    distance=5,
    rng=None,
    max_pos_attempts=2000,
    max_neg_attempts=2000,
    num_random_fg_points=8,  # new: extra points anywhere in mask
):
    """
    Generate positive points near centroid and optionally randomly within the mask.
    
    Args:
        mask: (H,W), foreground is mask>0
        centroid_xy: (cx, cy)
        num_pos_points: points around centroid
        num_neg_points: points in background
        distance: max radius around centroid
        num_random_fg_points: additional random points anywhere in mask
        
    Returns:
        pos_points: list[(x,y)]
        neg_points: list[(x,y)]
    """
    if rng is None:
        rng = np.random.default_rng()

    mask = (mask > 0).astype(np.uint8)
    H, W = mask.shape

    # ----------------------------
    # Positive points near centroid
    # ----------------------------
    cx, cy = float(centroid_xy[0]), float(centroid_xy[1])
    cx_i = int(np.clip(round(cx), 0, W-1))
    cy_i = int(np.clip(round(cy), 0, H-1))

    pos_points = []
    if mask[cy_i, cx_i] > 0:
        pos_points.append((cx_i, cy_i))

    attempts = 0
    while len(pos_points) < num_pos_points and attempts < max_pos_attempts:
        angle = rng.uniform(0, 2*np.pi)
        r = rng.uniform(0, distance)
        x = int(round(cx + r * np.cos(angle)))
        y = int(round(cy + r * np.sin(angle)))
        x = np.clip(x, 0, W-1)
        y = np.clip(y, 0, H-1)
        if mask[y, x] > 0 and (x, y) not in pos_points:
            pos_points.append((x, y))
        attempts += 1

    # fallback: random foreground if still short
    if len(pos_points) < num_pos_points:
        fg = np.argwhere(mask > 0)
        if len(fg) > 0:
            need = num_pos_points - len(pos_points)
            pick = fg[rng.integers(0, len(fg), size=need)]
            for y, x in pick:
                xy = (int(x), int(y))
                if xy not in pos_points:
                    pos_points.append(xy)

    # ----------------------------
    # Additional random points anywhere in mask
    # ----------------------------
    if num_random_fg_points > 0:
        fg = np.argwhere(mask > 0)
        if len(fg) > 0:
            pick = fg[rng.integers(0, len(fg), size=num_random_fg_points)]
            for y, x in pick:
                xy = (int(x), int(y))
                if xy not in pos_points:
                    pos_points.append(xy)

    # ----------------------------
    # Negative points
    # ----------------------------
    neg_points = []
    attempts = 0
    while len(neg_points) < num_neg_points and attempts < max_neg_attempts:
        x = int(rng.integers(0, W))
        y = int(rng.integers(0, H))
        if mask[y, x] == 0 and (x, y) not in neg_points:
            neg_points.append((x, y))
        attempts += 1

    return pos_points, neg_points

# -------------------------
# Dataset
# -------------------------
class FluorescenceDataset(Dataset):
    def __init__(
        self,
        images_hw,            # (N,H,W)
        masks_hw,             # (N,H,W)
        centroids_obj,        # object array, centroids_obj[i][channel] -> (K,2)
        channel_idx: int,
        num_pos_points=1,
        num_neg_points=1,
        distance=5,
        seed=42,
    ):
        assert len(images_hw) == len(masks_hw) == len(centroids_obj)
        self.images = images_hw
        self.masks = masks_hw
        self.centroids = centroids_obj
        self.channel_idx = channel_idx
        self.num_pos_points = num_pos_points
        self.num_neg_points = num_neg_points
        self.distance = distance
        self.seed = seed

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        img = np.clip(self.images[idx], 0, 255).astype(np.uint8)       # (H,W)
        img_rgb = np.repeat(img[..., None], 3, axis=-1)                # (H,W,3) uint8

        mask = self.masks[idx].astype(np.uint8)                        # (H,W)
        channel_centroids = self.centroids[idx][self.channel_idx]      # (K,2) or empty

        rng = np.random.default_rng(self.seed + idx)

        all_pos, all_neg = [], []
        if channel_centroids is not None and len(channel_centroids) > 0:
            for c in channel_centroids:
                cx, cy = float(c[0]), float(c[1])
                pos_points, neg_points = generate_points_around_centroid(
                                            mask,
                                            centroid_xy=(cx, cy),
                                            num_pos_points=8,
                                            num_neg_points=8,
                                            distance=5,
                                            num_random_fg_points=8  # extra points anywhere in blob
                                        )

                all_pos.extend(pos_points)
                all_neg.extend(neg_points)

        points = np.asarray(all_pos + all_neg, dtype=np.float32)  # (N,2) in (x,y)
        labels = np.ones((len(points),), dtype=np.int32)
        if len(all_neg) > 0:
            labels[len(all_pos):] = 0

        return {
            "image": img_rgb,     # NumPy HWC uint8
            "mask": mask,         # NumPy HW uint8 (optional, for eval/vis)
            "points": points,     # NumPy (N,2) float32
            "labels": labels,     # NumPy (N,) int32
        }


def collate_single(batch):
    # batch is a list of length batch_size; keep NumPy arrays as-is
    return batch[0]

# -------------------------
# Segmentation loop
# -------------------------
@torch.inference_mode()
def segment(dataloader, predictor: SAM2ImagePredictor):
    preds = []
    for sample in tqdm(dataloader, leave=True):
        img = sample["image"]        # NumPy HWC uint8
        pts = sample["points"]       # NumPy (N,2)
        lbl = sample["labels"]       # NumPy (N,)

        h, w = img.shape[:2]
        if pts.shape[0] == 0:
            preds.append(np.zeros((h, w), dtype=np.uint8))
            continue

        predictor.set_image(img)
        masks, scores, logits = predictor.predict(
            point_coords=pts,
            point_labels=lbl,
            multimask_output=False,
        )

        # Be robust to torch/numpy returns
        if torch.is_tensor(masks):
            masks = masks.detach().cpu().numpy()

        # Expect single mask when multimask_output=False
        pred = masks[0].astype(np.uint8) if masks.ndim == 3 else masks.astype(np.uint8)
        preds.append(pred)

    return np.stack(preds, axis=0)

In [3]:
from concurrent.futures import ThreadPoolExecutor
import threading
from typing import List, Tuple

# -------------------------
# Thread-safe segment wrapper
# -------------------------
def segment_threadsafe(
    dataloader, 
    predictor: SAM2ImagePredictor, 
    thread_id: int
) -> np.ndarray:
    """Thread-safe version - each thread gets its own predictor."""
    print(f"Thread {thread_id} starting...")
    local_predictor = SAM2ImagePredictor(predictor.model)  # Fresh predictor per thread
    
    preds = []
    for i, sample in enumerate(tqdm(dataloader, desc=f"Thread {thread_id}", leave=False)):
        img = sample["image"]
        pts = sample["points"]
        lbl = sample["labels"]
        
        h, w = img.shape[:2]
        if pts.shape[0] == 0:
            preds.append(np.zeros((h, w), dtype=np.uint8))
            continue
            
        local_predictor.set_image(img)
        masks, _, _ = local_predictor.predict(
            point_coords=pts,
            point_labels=lbl,
            multimask_output=False,
        )
        
        pred = masks[0].astype(np.uint8) if masks.ndim == 3 else masks.astype(np.uint8)
        preds.append(pred)
    
    print(f"Thread {thread_id} finished: {len(preds)} predictions")
    return np.stack(preds)

# -------------------------
# Parallel segmentation
# -------------------------
dataset_ch0 = FluorescenceDataset(
    images_clean[:, 0], mask_clean[:, 0], centroids_final, 
    channel_idx=0, num_pos_points=3, num_neg_points=50, distance=3
)
dataset_ch1 = FluorescenceDataset(
    images_clean[:, 1], mask_clean[:, 1], centroids_final, 
    channel_idx=1, num_pos_points=3, num_neg_points=50, distance=3
)

loader_ch0 = DataLoader(dataset_ch0, batch_size=1, shuffle=False, collate_fn=collate_single, num_workers=0)
loader_ch1 = DataLoader(dataset_ch1, batch_size=1, shuffle=False, collate_fn=collate_single, num_workers=0)

# Run in parallel (2 threads)
with ThreadPoolExecutor(max_workers=2) as executor:
    futures = [
        executor.submit(segment_threadsafe, loader_ch0, predictor, 0),
        executor.submit(segment_threadsafe, loader_ch1, predictor, 1)
    ]
    
    predictions_ch0, predictions_ch1 = [future.result() for future in futures]

print("Predictions CH0:", predictions_ch0.shape)
print("Predictions CH1:", predictions_ch1.shape)


Thread 0 starting...
Thread 1 starting...


Thread 0:   0%|          | 0/2697 [00:00<?, ?it/s]

Thread 1:   0%|          | 0/2697 [00:00<?, ?it/s]

Thread 0 finished: 2697 predictions
Thread 1 finished: 2697 predictions
Predictions CH0: (2697, 65, 65)
Predictions CH1: (2697, 65, 65)


In [ ]:
import matplotlib.pyplot as plt
from ipywidgets import interact, IntSlider
import numpy as np
from skimage import exposure

def normalize_contrast(img, pmin=2, pmax=98):
    """Stretch contrast to [pmin,pmax] percentiles."""
    pmin_val, pmax_val = np.percentile(img, (pmin, pmax))
    return exposure.rescale_intensity(img, in_range=(pmin_val, pmax_val), out_range=(0, 255)).astype(np.uint8)

def visualize_image(idx):
    plt.figure(figsize=(24, 10))
    plt.tight_layout(pad=2.0)

    # ----------------------------
    # Channel 0
    # ----------------------------
    img0_raw = images_clean[idx, 0]
    img0_norm = normalize_contrast(img0_raw)
    
    # Image with centroids
    img0_rgb = np.stack([img0_norm] * 3, axis=-1)
    for xy in centroids_final[idx][0]:
        x, y = int(round(xy[0])), int(round(xy[1]))
        if 0 <= y < img0_rgb.shape[0] and 0 <= x < img0_rgb.shape[1]:
            img0_rgb[y, x] = [0, 255, 0]  # bright green centroid

    # Predicted overlay
    ov0 = img0_rgb.copy()
    pred_mask0 = predictions_ch0[idx] > 0
    ov0[pred_mask0] = [255, 0, 0]  # red mask overlay

    # Ground truth overlay
    gt_mask0 = mask_clean[idx, 0] > 0
    gt0 = img0_rgb.copy()
    gt0[gt_mask0] = [255, 255, 0]  # yellow GT mask

    # Plot CH0
    plt.subplot(2, 5, 1)
    plt.imshow(img0_raw, cmap='gray')
    plt.title("CH0 Raw")
    plt.axis('off')

    plt.subplot(2, 5, 2)
    plt.imshow(img0_norm, cmap='gray')
    plt.title("CH0 Normalized")
    plt.axis('off')

    plt.subplot(2, 5, 3)
    plt.imshow(img0_rgb)
    plt.title("CH0 Image + Centroids")
    plt.axis('off')

    plt.subplot(2, 5, 4)
    plt.imshow(gt0)
    plt.title("CH0 GT Mask Overlay")
    plt.axis('off')

    plt.subplot(2, 5, 5)
    plt.imshow(ov0)
    plt.title("CH0 Pred Mask Overlay")
    plt.axis('off')

    # ----------------------------
    # Channel 1
    # ----------------------------
    img1_raw = images_clean[idx, 1]
    img1_norm = normalize_contrast(img1_raw)
    
    # Image with centroids
    img1_rgb = np.stack([img1_norm] * 3, axis=-1)
    for xy in centroids_final[idx][1]:
        x, y = int(round(xy[0])), int(round(xy[1]))
        if 0 <= y < img1_rgb.shape[0] and 0 <= x < img1_rgb.shape[1]:
            img1_rgb[y, x] = [0, 255, 0]  # bright green centroid

    # Predicted overlay
    ov1 = img1_rgb.copy()
    pred_mask1 = predictions_ch1[idx] > 0
    ov1[pred_mask1] = [255, 0, 255]  # magenta mask overlay

    # Ground truth overlay
    gt_mask1 = mask_clean[idx, 1] > 0
    gt1 = img1_rgb.copy()
    gt1[gt_mask1] = [255, 255, 0]  # yellow GT mask

    # Plot CH1
    plt.subplot(2, 5, 6)
    plt.imshow(img1_raw, cmap='gray')
    plt.title("CH1 Raw")
    plt.axis('off')

    plt.subplot(2, 5, 7)
    plt.imshow(img1_norm, cmap='gray')
    plt.title("CH1 Normalized")
    plt.axis('off')

    plt.subplot(2, 5, 8)
    plt.imshow(img1_rgb)
    plt.title("CH1 Image + Centroids")
    plt.axis('off')

    plt.subplot(2, 5, 9)
    plt.imshow(gt1)
    plt.title("CH1 GT Mask Overlay")
    plt.axis('off')

    plt.subplot(2, 5, 10)
    plt.imshow(ov1)
    plt.title("CH1 Pred Mask Overlay")
    plt.axis('off')

    plt.suptitle(f"Image {idx}: SAM2 Segmentation Results", fontsize=16, y=0.98)
    plt.show()

# Interactive slider (uncomment to use full dataset)
interact(
    visualize_image,
    idx=IntSlider(min=0, max=min(1000, len(images_clean)-1), step=1, value=0)
)


NameError: name 'df' is not defined

In [ ]:
predictions_combined = np.stack([predictions_ch0, predictions_ch1], axis=1)  # (N,2,H,W)
np.save("../data/prompts/predictions_sam2.npy", predictions_combined)